# Liquid Analysis Visualization
Visualizes results from `liquid_analysis.py` — combining YOLO-NAS and Green HSV detection.

**Data source:** `analysis_result/liquid_analysis.csv`

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
import os
from pathlib import Path

plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

# Load data
CSV_PATH = "./analysis_result/liquid_analysis.csv"
IMAGE_DIR = "./analysis_result"

df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
df.head()

## 1. Overview: Liquid Detection Accuracy

Compare expected vs detected liquid presence across all 4 action types.

In [ ]:
expected = {
    "before_aspirate": "NO",
    "after_aspirate": "YES",
    "before_dispense": "YES",
    "after_dispense": "NO",
}

df["expected"] = df["action"].map(expected)
df["correct"] = df["liquid_detected"] == df["expected"]

# Accuracy per action
accuracy = df.groupby("action")["correct"].mean() * 100
accuracy = accuracy.reindex(["before_aspirate", "after_aspirate", "before_dispense", "after_dispense"])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart: accuracy per action
colors = ["#2ecc71" if v == 100 else "#e74c3c" for v in accuracy.values]
bars = axes[0].bar(accuracy.index, accuracy.values, color=colors, edgecolor="black", linewidth=0.5)
axes[0].set_ylim(0, 110)
axes[0].set_ylabel("Accuracy (%)")
axes[0].set_title("Detection Accuracy by Action")
axes[0].set_xticklabels(accuracy.index, rotation=20, ha="right")
for bar, val in zip(bars, accuracy.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                 f"{val:.0f}%", ha="center", fontweight="bold")

# Confusion-style table
actions = ["before_aspirate", "after_aspirate", "before_dispense", "after_dispense"]
table_data = []
for action in actions:
    sub = df[df["action"] == action]
    total = len(sub)
    correct_n = sub["correct"].sum()
    table_data.append([action, expected[action], f"{correct_n}/{total}", f"{correct_n/total*100:.0f}%"])

axes[1].axis("off")
table = axes[1].table(
    cellText=table_data,
    colLabels=["Action", "Expected", "Correct", "Accuracy"],
    loc="center",
    cellLoc="center",
)
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.2, 1.8)

# Color header
for j in range(4):
    table[0, j].set_facecolor("#34495e")
    table[0, j].set_text_props(color="white", fontweight="bold")

plt.suptitle("Liquid Detection Results", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

overall = df["correct"].mean() * 100
print(f"\nOverall Accuracy: {df['correct'].sum()}/{len(df)} ({overall:.1f}%)")

## 2. YOLO Fill Ratio vs Green Ratio Across Columns

Shows how consistently each column was filled. Each column represents one transfer (A1 through A10+).

In [ ]:
# Compare YOLO fill and green ratio for after_aspirate (liquid in tips)
aspirated = df[df["action"] == "after_aspirate"].copy()
aspirated = aspirated.sort_values("column", key=lambda x: x.str.extract(r'(\d+)')[0].astype(int))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# YOLO fill ratio per column
x = range(len(aspirated))
axes[0].bar(x, aspirated["avg_yolo_fill"].astype(float), color="#3498db", edgecolor="black", linewidth=0.5)
axes[0].set_xticks(x)
axes[0].set_xticklabels(aspirated["column"], rotation=0)
axes[0].set_ylabel("Avg YOLO Fill Ratio")
axes[0].set_xlabel("Column")
axes[0].set_title("YOLO Fill Ratio After Aspirate")
axes[0].set_ylim(0, 0.5)
axes[0].axhline(y=aspirated["avg_yolo_fill"].astype(float).mean(), color="red", linestyle="--", label=f"Mean: {aspirated['avg_yolo_fill'].astype(float).mean():.3f}")
axes[0].legend()

# Green ratio per column
axes[1].bar(x, aspirated["avg_green"].astype(float), color="#2ecc71", edgecolor="black", linewidth=0.5)
axes[1].set_xticks(x)
axes[1].set_xticklabels(aspirated["column"], rotation=0)
axes[1].set_ylabel("Avg Green Ratio")
axes[1].set_xlabel("Column")
axes[1].set_title("Green Detection Ratio After Aspirate")
axes[1].set_ylim(0, 0.15)
axes[1].axhline(y=aspirated["avg_green"].astype(float).mean(), color="red", linestyle="--", label=f"Mean: {aspirated['avg_green'].astype(float).mean():.4f}")
axes[1].legend()

plt.suptitle("Liquid Measurement Consistency Across Columns (After Aspirate)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

# Stats
print(f"YOLO Fill — Mean: {aspirated['avg_yolo_fill'].astype(float).mean():.4f}, Std: {aspirated['avg_yolo_fill'].astype(float).std():.4f}, CV: {aspirated['avg_yolo_fill'].astype(float).std()/aspirated['avg_yolo_fill'].astype(float).mean()*100:.1f}%")
print(f"Green     — Mean: {aspirated['avg_green'].astype(float).mean():.4f}, Std: {aspirated['avg_green'].astype(float).std():.4f}, CV: {aspirated['avg_green'].astype(float).std()/aspirated['avg_green'].astype(float).mean()*100:.1f}%")

## 3. Per-Tip Uniformity (Tip 1–8)

Are all 8 tips picking up the same amount of liquid? This checks tip-to-tip consistency within each image.

In [ ]:
# Per-tip green ratio for all after_aspirate images
aspirated = df[df["action"] == "after_aspirate"].copy()
aspirated = aspirated.sort_values("column", key=lambda x: x.str.extract(r'(\d+)')[0].astype(int))

tip_green_cols = [f"tip_{i}_green_ratio" for i in range(1, 9)]
tip_fill_cols = [f"tip_{i}_yolo_fill" for i in range(1, 9)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Green ratio heatmap
green_data = aspirated[tip_green_cols].astype(float).values
im1 = axes[0].imshow(green_data, cmap="Greens", aspect="auto", vmin=0, vmax=0.15)
axes[0].set_yticks(range(len(aspirated)))
axes[0].set_yticklabels(aspirated["column"].values)
axes[0].set_xticks(range(8))
axes[0].set_xticklabels([f"Tip {i}" for i in range(1, 9)], rotation=45, ha="right")
axes[0].set_title("Green Ratio per Tip (After Aspirate)")
axes[0].set_ylabel("Column")
plt.colorbar(im1, ax=axes[0], label="Green Ratio")

# Add text values
for i in range(green_data.shape[0]):
    for j in range(green_data.shape[1]):
        axes[0].text(j, i, f"{green_data[i,j]:.3f}", ha="center", va="center", fontsize=8,
                     color="white" if green_data[i,j] > 0.08 else "black")

# YOLO fill heatmap
fill_data = aspirated[tip_fill_cols].astype(float).values
im2 = axes[1].imshow(fill_data, cmap="Blues", aspect="auto", vmin=0, vmax=0.5)
axes[1].set_yticks(range(len(aspirated)))
axes[1].set_yticklabels(aspirated["column"].values)
axes[1].set_xticks(range(8))
axes[1].set_xticklabels([f"Tip {i}" for i in range(1, 9)], rotation=45, ha="right")
axes[1].set_title("YOLO Fill Ratio per Tip (After Aspirate)")
axes[1].set_ylabel("Column")
plt.colorbar(im2, ax=axes[1], label="Fill Ratio")

for i in range(fill_data.shape[0]):
    for j in range(fill_data.shape[1]):
        axes[1].text(j, i, f"{fill_data[i,j]:.2f}", ha="center", va="center", fontsize=8,
                     color="white" if fill_data[i,j] > 0.3 else "black")

plt.suptitle("Tip-to-Tip Uniformity Heatmaps", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

# Per-tip stats
print("Per-tip average green ratio (across all columns):")
for i in range(1, 9):
    col = f"tip_{i}_green_ratio"
    vals = aspirated[col].astype(float)
    print(f"  Tip {i}: mean={vals.mean():.4f}, std={vals.std():.4f}")

## 4. Before vs After Comparison

Shows the signal change between before and after aspirate/dispense. A good protocol should show a clear jump.

In [ ]:
# Group by action and show distributions
actions_order = ["before_aspirate", "after_aspirate", "before_dispense", "after_dispense"]
action_labels = ["Before\nAspirate", "After\nAspirate", "Before\nDispense", "After\nDispense"]
action_colors = ["#ecf0f1", "#2ecc71", "#2ecc71", "#ecf0f1"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Green ratio by action
green_by_action = [df[df["action"] == a]["avg_green"].astype(float).values for a in actions_order]
bp1 = axes[0].boxplot(green_by_action, labels=action_labels, patch_artist=True, widths=0.6)
for patch, color in zip(bp1["boxes"], action_colors):
    patch.set_facecolor(color)
axes[0].set_ylabel("Avg Green Ratio")
axes[0].set_title("Green Detection by Action")
axes[0].axhline(y=0.02, color="red", linestyle="--", alpha=0.5, label="Threshold (0.02)")
axes[0].legend()

# YOLO fill by action
fill_by_action = [df[df["action"] == a]["avg_yolo_fill"].astype(float).values for a in actions_order]
bp2 = axes[1].boxplot(fill_by_action, labels=action_labels, patch_artist=True, widths=0.6)
for patch, color in zip(bp2["boxes"], action_colors):
    patch.set_facecolor(color)
axes[1].set_ylabel("Avg YOLO Fill Ratio")
axes[1].set_title("YOLO Fill Ratio by Action")

plt.suptitle("Signal Separation: Empty vs Liquid", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

# Separation stats
for metric, name in [("avg_green", "Green"), ("avg_yolo_fill", "YOLO Fill")]:
    empty = df[df["action"].isin(["before_aspirate", "after_dispense"])][metric].astype(float)
    liquid = df[df["action"].isin(["after_aspirate", "before_dispense"])][metric].astype(float)
    gap = liquid.min() - empty.max()
    print(f"{name}: empty max={empty.max():.4f}, liquid min={liquid.min():.4f}, gap={gap:.4f} ({'CLEAR' if gap > 0 else 'OVERLAP'})")

## 5. YOLO vs Green: Method Comparison

Scatter plot comparing the two detection methods. Points should cluster in corners (both agree) if both methods work well.

In [ ]:
# Scatter: YOLO fill vs Green ratio
fig, ax = plt.subplots(figsize=(8, 6))

colors_map = {
    "before_aspirate": "#95a5a6",
    "after_aspirate": "#2ecc71",
    "before_dispense": "#3498db",
    "after_dispense": "#e74c3c",
}

for action, group in df.groupby("action"):
    ax.scatter(
        group["avg_yolo_fill"].astype(float),
        group["avg_green"].astype(float),
        label=action,
        color=colors_map[action],
        s=80, edgecolors="black", linewidth=0.5, alpha=0.8,
    )

ax.set_xlabel("Avg YOLO Fill Ratio")
ax.set_ylabel("Avg Green Ratio")
ax.set_title("YOLO vs Green Detection — Method Agreement")
ax.legend(loc="upper left")
ax.axhline(y=0.02, color="green", linestyle="--", alpha=0.3, label="Green threshold")
ax.axvline(x=0.05, color="blue", linestyle="--", alpha=0.3, label="YOLO threshold")

plt.tight_layout()
plt.show()

# Agreement rate
df["yolo_says"] = df["yolo_liquid_tips"].astype(int) > 0
df["green_says"] = df["avg_green"].astype(float) > 0.02
agreement = (df["yolo_says"] == df["green_says"]).mean() * 100
print(f"YOLO and Green agree on {agreement:.1f}% of images")

## 6. Timeline View

Shows YOLO fill and green ratio over time (image sequence) to see the aspirate/dispense cycle pattern.

In [ ]:
# Timeline: signal over image sequence
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

x = range(len(df))
action_colors_seq = [colors_map[a] for a in df["action"]]

# Green ratio timeline
axes[0].bar(x, df["avg_green"].astype(float), color=action_colors_seq, edgecolor="black", linewidth=0.3)
axes[0].axhline(y=0.02, color="red", linestyle="--", alpha=0.5)
axes[0].set_ylabel("Avg Green Ratio")
axes[0].set_title("Green Detection Over Time")

# YOLO fill timeline
axes[1].bar(x, df["avg_yolo_fill"].astype(float), color=action_colors_seq, edgecolor="black", linewidth=0.3)
axes[1].set_ylabel("Avg YOLO Fill")
axes[1].set_title("YOLO Fill Ratio Over Time")
axes[1].set_xlabel("Image Index")

# Add column labels at bottom
for i, (_, row) in enumerate(df.iterrows()):
    if row["action"] == "before_aspirate":
        axes[1].axvline(x=i-0.5, color="gray", linestyle=":", alpha=0.3)
        axes[1].text(i+1.5, axes[1].get_ylim()[1]*0.9, row["column"],
                     ha="center", fontsize=8, color="gray")

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=a) for a, c in colors_map.items()]
axes[0].legend(handles=legend_elements, loc="upper right", fontsize=9, ncol=2)

plt.tight_layout()
plt.show()

## 7. Sample Annotated Images

Side-by-side comparison of before/after aspirate for the first column.

In [ ]:
# Show annotated images: before vs after aspirate for first 3 columns
cols_to_show = ["A1", "A2", "A3"]
fig, axes = plt.subplots(len(cols_to_show), 2, figsize=(14, 5 * len(cols_to_show)))

for row_idx, col in enumerate(cols_to_show):
    for col_idx, action in enumerate(["before_aspirate", "after_aspirate"]):
        sub = df[(df["action"] == action) & (df["column"] == col)]
        if len(sub) == 0:
            continue
        fname = sub.iloc[0]["filename"]
        img_path = os.path.join(IMAGE_DIR, f"annotated_{fname}")
        if os.path.exists(img_path):
            img = mpimg.imread(img_path)
            axes[row_idx, col_idx].imshow(img)
            verdict = sub.iloc[0]["liquid_detected"]
            axes[row_idx, col_idx].set_title(f"{action} — {col} — {verdict}", fontsize=11)
        axes[row_idx, col_idx].axis("off")

plt.suptitle("Annotated Image Comparison: Before vs After Aspirate", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 8. Summary Statistics

In [ ]:
# Summary statistics table
print("=" * 70)
print("FULL SUMMARY")
print("=" * 70)

summary_data = []
for action in actions_order:
    sub = df[df["action"] == action]
    n = len(sub)
    correct_n = sub["correct"].sum()
    avg_fill = sub["avg_yolo_fill"].astype(float).mean()
    avg_green = sub["avg_green"].astype(float).mean()
    std_fill = sub["avg_yolo_fill"].astype(float).std()
    std_green = sub["avg_green"].astype(float).std()
    
    summary_data.append({
        "Action": action,
        "Expected": expected[action],
        "Accuracy": f"{correct_n}/{n} ({correct_n/n*100:.0f}%)",
        "Avg YOLO Fill": f"{avg_fill:.4f} +/- {std_fill:.4f}",
        "Avg Green": f"{avg_green:.4f} +/- {std_green:.4f}",
    })

summary_df = pd.DataFrame(summary_data)
display(summary_df.style.set_properties(**{"text-align": "center"}).hide(axis="index"))

print(f"\nOverall Detection Accuracy: {df['correct'].sum()}/{len(df)} ({df['correct'].mean()*100:.1f}%)")
print(f"YOLO-Green Agreement: {(df['yolo_says'] == df['green_says']).mean()*100:.1f}%")

# Aspirate consistency (CV = coefficient of variation)
asp = df[df["action"] == "after_aspirate"]
fill_cv = asp["avg_yolo_fill"].astype(float).std() / asp["avg_yolo_fill"].astype(float).mean() * 100
green_cv = asp["avg_green"].astype(float).std() / asp["avg_green"].astype(float).mean() * 100
print(f"\nAspirate Consistency (lower CV = more consistent):")
print(f"  YOLO Fill CV: {fill_cv:.1f}%")
print(f"  Green CV:     {green_cv:.1f}%")